<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day12-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 12 lab: AlphaFold in practice {.unnumbered}

Most of this lab happens in web tools: the **AlphaFold Server**
(alphafoldserver.com, AlphaFold3), the **AlphaFold Protein Structure
Database** (alphafold.ebi.ac.uk) and PyMOL. This notebook is the helper:

- **Part A** fetches the chain sequences of four target complexes from the
  PDB, ready to paste into the AlphaFold Server. There is one target per
  "region" of the lecture's framework for where structure prediction
  works and where it does not.
- **Part B** reads AlphaFold DB models: per-residue confidence (pLDDT),
  the predicted aligned error (PAE), hydrogen bonds, and a comparison with
  an experimental structure.
- **Part C** reads the results `.zip` that the AlphaFold Server gives you:
  pTM, ipTM, the PAE heatmap, and the interface score **ipSAE**.
- **Part D** pools the class's results.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

**The four targets** (one per region):

| Region | What makes it hard | Target | PDB |
|---|---|---|---|
| I, tractable | nothing: stable interface, strong co-evolution | Ephrin-A5 : EphB2 | 1SHW |
| II, ambiguity and competition | a plausible interface is easy; the *right partner* is hard | Izumo1 : Juno | 5F4E |
| III, missing co-evolution | no joint evolutionary pressure between the partners | antibody : antigen | 8TQ7 |
| IV, missing context and dynamics | disorder-to-order folding on binding | p53 transactivation peptide : MDM2 | 1YCR |

The AlphaFold Server allows 20 jobs per day and account: split the four
targets across your group.

In [ ]:
import os, io, re, json, glob, zipfile, tempfile, subprocess, sys
import numpy as np
import requests
import matplotlib.pyplot as plt

WORK = tempfile.mkdtemp(prefix="kb8029_day12_")      # all downloads go here, not next to the notebook
print("working directory for downloads:", WORK)

try:
    import tmtools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tmtools"], check=True)
    import tmtools
from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB.Polypeptide import three_to_index, index_to_one

def todo(what):
    '''Placeholder for code you write: replace the whole todo(...) call with your own code.'''
    raise NotImplementedError(f"TODO in this cell: {what}")

## Part A: get the sequences

The RCSB PDB serves every entry's sequences as FASTA. Each record is one
*entity* (a distinct chain type). Its header lists which chains in the
file carry that sequence, for example `Chains A, C`. For the AlphaFold
Server, add one **Protein** entity per distinct sequence, with *copies*
set to 1 (we predict one copy of each complex).

In [ ]:
TARGETS = {"I": ("Ephrin-A5 : EphB2", "1SHW"), "II": ("Izumo1 : Juno", "5F4E"),
           "III": ("antibody : antigen", "8TQ7"), "IV": ("p53 TAD : MDM2", "1YCR")}

def pdb_entities(pdb_id):
    '''Return a list of (header, sequence) pairs, one per entity of a PDB entry.'''
    text = requests.get(f"https://www.rcsb.org/fasta/entry/{pdb_id}", timeout=60).text
    records = []
    for block in text.strip().split(">")[1:]:
        header, *seq_lines = block.strip().split("\n")
        records.append((header, "".join(seq_lines)))
    return records

entities = {}
for region, (name, pdb_id) in TARGETS.items():
    entities[region] = pdb_entities(pdb_id)
    print(f"Region {region}: {name} ({pdb_id})")
    for header, seq in entities[region]:
        n_res = todo("the number of residues in this entity's sequence")
        print(f"   {header[:90]}   [{n_res} residues]")

In [ ]:
MY_REGION = "I"     # change to the region your group was assigned
print(f"Paste these into the AlphaFold Server, one Protein entity each (copies = 1):\n")
for header, seq in entities[MY_REGION]:
    print(f">{header}\n{seq}\n")

## Part B: reading AlphaFold DB models

The AlphaFold DB API returns, for a UniProt accession, links to the model
(its B-factor column holds **pLDDT**, 0-100) and to the **PAE** matrix.
pLDDT > 90 is very high confidence, 70-90 confident, 50-70 low, and below
50 very low, which usually means disordered.

In [ ]:
def afdb_entry(accession):
    '''Metadata, parsed structure and per-residue pLDDT of the AlphaFold DB model for a UniProt accession.'''
    meta = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{accession}", timeout=60).json()[0]
    pdb_text = requests.get(meta["pdbUrl"], timeout=60).text
    structure = PDBParser(QUIET=True).get_structure(accession, io.StringIO(pdb_text))
    plddt = np.array([res["CA"].get_bfactor() for res in structure.get_residues()])
    return meta, structure, plddt

meta, s_p94485, plddt = afdb_entry("P94485")
print(meta["uniprotDescription"], "|", len(plddt), "residues | model version", meta["latestVersion"])
pae = np.array(requests.get(meta["paeDocUrl"], timeout=60).json()[0]["predicted_aligned_error"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(np.arange(1, len(plddt) + 1), plddt)
for y, c in [(90, "#2f6db5"), (70, "#7fb3e6"), (50, "#f0a030")]:
    a1.axhline(y, color=c, ls="--", lw=0.8)
a1.set_xlabel("residue"); a1.set_ylabel("pLDDT"); a1.set_ylim(0, 100); a1.set_title("P94485: pLDDT per residue")
im = a2.imshow(pae, cmap="Greens_r", vmin=0, vmax=30)
a2.set_xlabel("aligned on residue"); a2.set_ylabel("error of residue"); a2.set_title("P94485: PAE (Å)")
plt.colorbar(im, ax=a2, fraction=0.046); plt.tight_layout(); plt.show()

**Hydrogen bonds.** A hydrogen bond needs a donor and an acceptor (N or O
atoms) about 2.6-3.5 Å apart. List every N or O atom of *another* residue
within 3.5 Å of an N or O atom of Ser 11 in P94485. Most partners are
Ser 11's neighbours in the helix (i ± 1 to i ± 4); look for the one that
is far away in sequence.

In [ ]:
atoms = list(s_p94485.get_atoms())
search = NeighborSearch(atoms)
ser11 = [res for res in s_p94485.get_residues() if res.id[1] == 11][0]
print("residue 11 is", ser11.get_resname())
for atom in ser11:
    if atom.element not in ("N", "O"):
        continue
    for other in search.search(atom.coord, 3.5):
        partner = other.get_parent()
        if other.element in ("N", "O") and partner is not ser11:
            d = todo('the distance in Å between atom and other')
            print(f"  SER 11 {atom.get_id():3s} -- {partner.get_resname()} {partner.id[1]:3d} {other.get_id():4s}  {d:.2f} Å"
                  f"   (sequence separation {abs(partner.id[1] - 11)})")

**Disorder.** Which of these four models is the most disordered? Compare
the fraction of residues with very low confidence (pLDDT < 50), which the
AlphaFold DB API also reports directly.

In [ ]:
for acc in ["A8MVM7", "Q96NG5", "Q8IYP9", "Q8W3K0"]:
    m, _, p = afdb_entry(acc)
    frac_low = todo('fraction of residues with pLDDT below 50')
    print(f"{acc}: {m['uniprotDescription'][:45]:45s} {len(p):5d} residues, mean pLDDT {p.mean():5.1f}, "
          f"pLDDT < 50: {frac_low:.2f}  (API: {m['fractionPlddtVeryLow']:.2f})")

m, _, p = afdb_entry("Q00768")
print(f"\nQ00768 ({m['uniprotDescription']}): mean pLDDT first 30 residues {p[:30].mean():.1f}, "
      f"last 30 residues {p[-30:].mean():.1f}")

**Model vs. experiment.** The AlphaFold DB model of mouse Ephrin-A5
(UniProt O08543) against the crystal structure of the same domain in
1SHW, chain A. TM-align superimposes the two and reports the RMSD over
the aligned Cα atoms and the TM-score (0 to 1; above about 0.5 means the
same fold).

In [ ]:
def ca_chain(structure, chain_id=None):
    chain = structure[0][chain_id] if chain_id else next(iter(structure[0]))
    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    seq = "".join(index_to_one(three_to_index(r.get_resname())) for r in residues)
    return np.array([r["CA"].coord for r in residues]), seq

_, s_model, _ = afdb_entry("O08543")
crystal = PDBParser(QUIET=True).get_structure("1SHW", io.StringIO(requests.get("https://files.rcsb.org/download/1SHW.pdb", timeout=60).text))
X_model, seq_model = ca_chain(s_model)
X_xtal, seq_xtal = ca_chain(crystal, "A")
result = tmtools.tm_align(X_model, X_xtal, seq_model, seq_xtal)
print(f"model {len(seq_model)} residues, crystal chain A {len(seq_xtal)} residues")
print(f"TM-score (normalised by the crystal chain) {result.tm_norm_chain2:.3f}, RMSD {result.rmsd:.2f} Å")

## Part C: your AlphaFold Server result

Download the result of your job from the AlphaFold Server (the `.zip`
button) and load it here. On Colab, run the cell and choose the file; on
your own computer, set `ZIP_PATH`. The zip holds five models
(`model_0` is the top-ranked), and for each a `summary_confidences`
file (pTM, ipTM, chain-pair ipTM) and a `full_data` file (the PAE matrix
and per-token chain IDs).

In [ ]:
ZIP_PATH = ''      # set a local path, or leave empty to upload on Colab
if not ZIP_PATH:
    try:
        from google.colab import files
        uploaded = files.upload()
        ZIP_PATH = os.path.join(WORK, next(iter(uploaded)))
        open(ZIP_PATH, "wb").write(uploaded[os.path.basename(ZIP_PATH)])
    except ImportError:
        raise SystemExit("Set ZIP_PATH to the AlphaFold Server .zip you downloaded.")
job_dir = os.path.join(WORK, os.path.basename(ZIP_PATH).replace(".zip", ""))
zipfile.ZipFile(ZIP_PATH).extractall(job_dir)
summary_file = sorted(glob.glob(os.path.join(job_dir, "*summary_confidences_0.json")))[0]
full_file = sorted(glob.glob(os.path.join(job_dir, "*full_data_0.json")))[0]
cif_file = sorted(glob.glob(os.path.join(job_dir, "*model_0.cif")))[0]
print("top-ranked model:", os.path.basename(cif_file))

In [ ]:
summary = json.load(open(summary_file))
ptm = todo('the pTM value from the summary dictionary')
iptm = todo('the ipTM value from the summary dictionary')
print(f"pTM {ptm:.2f}   ipTM {iptm}   ranking score {summary['ranking_score']:.2f}")
print("chain-pair ipTM:")
for row in summary["chain_pair_iptm"]:
    print("   ", row)

In [ ]:
full = json.load(open(full_file))
pae_af3 = np.array(full["pae"])
chains = np.array(full["token_chain_ids"])
boundaries = [i for i in range(1, len(chains)) if chains[i] != chains[i - 1]]
plt.figure(figsize=(5.5, 5))
plt.imshow(pae_af3, cmap="Greens_r", vmin=0, vmax=30)
for b in boundaries:
    plt.axhline(b - 0.5, color="red", lw=1); plt.axvline(b - 0.5, color="red", lw=1)
plt.colorbar(label="PAE (Å)", fraction=0.046)
plt.title("PAE of the top-ranked model; red lines = chain boundaries", fontsize=9)
plt.xlabel("aligned on token"); plt.ylabel("error of token"); plt.show()
print("tokens per chain:", {c: int((chains == c).sum()) for c in dict.fromkeys(chains)})

**ipSAE.** ipTM is computed over *all* residue pairs between two chains,
so long disordered tails or non-interacting domains pull it down (or add
noise) even when the interface itself is modelled well. **ipSAE**
(Dunbrack, 2025) uses only residue pairs whose PAE is below a cutoff.
The script is downloaded from its GitHub repository and run on the
top-ranked model with a PAE cutoff and a distance cutoff of 10 Å each.

In [ ]:
ipsae_script = os.path.join(WORK, "ipsae.py")
if not os.path.exists(ipsae_script):
    open(ipsae_script, "w").write(requests.get("https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py", timeout=60).text)
subprocess.run([sys.executable, ipsae_script, full_file, cif_file, "10", "10"], check=True, capture_output=True)
out_file = cif_file.replace(".cif", "_10_10.txt")
rows = [line.split() for line in open(out_file) if line.strip()]
header, rows = rows[0], rows[1:]
col = {name: k for k, name in enumerate(header)}
print(f"{'chains':8s} {'type':5s} {'ipSAE':>7s} {'ipTM_af':>8s} {'pDockQ':>7s}")
for r in rows:
    if len(r) > col["ipSAE"]:
        print(f"{r[col['Chn1']]}-{r[col['Chn2']]:5s} {r[col['Type']]:5s} {float(r[col['ipSAE']]):7.3f} "
              f"{float(r[col['ipTM_af']]):8.3f} {float(r[col['pDockQ']]):7.3f}")
ipsae_max = max(float(r[col["ipSAE"]]) for r in rows if len(r) > col["ipSAE"] and r[col["Type"]] == "max")
print(f"\nipSAE (max over chain pairs): {ipsae_max:.3f}")

## Part D: pooling the class's results

Fill in one row per group (region, pTM, ipTM, ipSAE of the top-ranked
model), then plot ipTM and ipSAE side by side for each region.

In [ ]:
class_results = [
    # region, ptm, iptm, ipsae   <- replace with the numbers from every group
    ("I", 0.0, 0.0, 0.0),
    ("II", 0.0, 0.0, 0.0),
    ("III", 0.0, 0.0, 0.0),
    ("IV", 0.0, 0.0, 0.0),
]
regions = [r[0] for r in class_results]
x = np.arange(len(regions))
plt.figure(figsize=(6, 3.8))
plt.bar(x - 0.2, [r[2] for r in class_results], width=0.4, label="ipTM")
plt.bar(x + 0.2, [r[3] for r in class_results], width=0.4, label="ipSAE")
plt.xticks(x, [f"Region {r}" for r in regions]); plt.ylim(0, 1); plt.legend()
plt.title("Interface confidence by region"); plt.show()

## Hand in

Upload this notebook (File → Download → .ipynb) and your PyMOL images
with the lab quiz on Canvas.